In [ ]:
# ── 패키지 설치 (Colab 초기 실행 시)
!pip install lightgbm catboost optuna koreanize-matplotlib --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder

try:
    import koreanize_matplotlib
except ImportError:
    pass

plt.rcParams['axes.unicode_minus'] = False
RANDOM_STATE = 42
TARGET = '임신 성공 여부'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 경로 설정 (Colab: '/content/train.csv', 로컬: 실제 경로)
DATA_DIR = '/content/drive/MyDrive/'
train_raw = pd.read_csv(DATA_DIR + 'train.csv')
test_raw  = pd.read_csv(DATA_DIR + 'test.csv')

train_raw.columns = [c.strip() for c in train_raw.columns]
test_raw.columns  = [c.strip() for c in test_raw.columns]

print(f'train: {train_raw.shape}  /  test: {test_raw.shape}')
print(f'타깃 분포:\n{train_raw[TARGET].value_counts(normalize=True).round(3)}')

In [ ]:
# ── 나이 구간   (하드코딩 → 데이터 통계 불필요)
# 만45-50세를 High_Reversal로 분리: 기증 난자 비율 증가로 성공률이 만43-44세보다 역전
AGE_INFO = {
    '만18-34세': {'median': 26.0,   'risk': 'Normal'},
    '만35-37세': {'median': 36.0,   'risk': 'High_Early'},
    '만38-39세': {'median': 38.5,   'risk': 'High_Early'},
    '만40-42세': {'median': 41.0,   'risk': 'High_Extreme'},
    '만43-44세': {'median': 43.5,   'risk': 'High_Extreme'},
    '만45-50세': {'median': 47.5,   'risk': 'High_Reversal'},  ## <<
    '알 수 없음': {'median': np.nan, 'risk': 'Unknown'},
}
AGE_MEDIAN_FILLNA = 38.5  # train median (하드코딩 → test 재사용)

# 결측 플래그(_isna)는 결측 자체가 '시술 미수행'이라는 신호일 때 만드는 변수
# 배아 해동 경과일: 결측=신선주기, 값있음=동결주기 → 강력한 시술 신호
# 난자 혼합 경과일: 결측=체외수정 미수행 신호
# 배아 이식 경과일: 결측=이식 미수행 신호
# '난자 해동 경과일'은 팀 토론에서 결측률 96% 이상으로 확인되어
#   삭제하기로 결정함 (COLS_TO_DROP에 포함)
# 96% 결측이면 플래그를 만들어도 96%가 0인 거의 상수 변수가 되어 모델에 의미 있는 신호를 주지 못함
# 정보량이 없는 변수는 오히려 모델 성능을 희석시킬 수 있음
# COLS_TO_DROP에서는 제거하지 않음 → 원본 컬럼 삭제는 그대로 유지
ISNA_FLAG_COLS = [
    '배아 해동 경과일',
    '난자 혼합 경과일',
    '배아 이식 경과일',
]

# ── 유전 검사: 99%+ 결측 = 대부분 미수행
GENETIC_TEST_COLS = ['PGS 시술 여부', 'PGD 시술 여부', '착상 전 유전 검사 사용 여부']

# ── 멀티라벨 (하드코딩 → test 신규 범주가 컬럼 구조를 바꾸지 않음)
KNOWN_REASONS = ['현재 시술용', '배아 저장용', '난자 저장용', '기증용', '연구용']

# ── 수치형 결측 대체값 (-999: 트리가 구조적 결측을 별도 분기로 학습)
NUMERIC_FILL = -999

# ── 로그 변환 대상 (왜도 큰 배아/난자 수치)
LOG_COLS = [
    '총 생성 배아 수', '미세주입된 난자 수', '미세주입에서 생성된 배아 수',
    '수집된 신선 난자 수', '혼합된 난자 수', '파트너 정자와 혼합된 난자 수',
]

# ── IQR 클리핑 대상 (상한값은 train에서 계산 후 아래 딕셔너리에 저장)
IQR_CLIP_COLS = ['저장된 배아 수', '미세주입 후 저장된 배아 수', '해동된 배아 수']
iqr_clip_bounds: dict = {}  # train fit 후 채워짐 → test transform에 재사용

# ── 삭제 컬럼 (결측 96%+ 또는 희소 변수)
COLS_TO_DROP = [
    'ID', '임신 시도 또는 마지막 임신 경과 연수', '난자 해동 경과일',
    '불임 원인 - 여성 요인', '불임 원인 - 자궁경부 문제',
    '불임 원인 - 정자 면역학적 요인', '불임 원인 - 정자 운동성',
    '불임 원인 - 정자 농도', '불임 원인 - 정자 형태',
]

print('상수 정의 완료')

In [ ]:
# train에서 IQR 상한값 계산 후 저장 (test에는 이 값을 그대로 사용)
for col in IQR_CLIP_COLS:
    if col in train_raw.columns:
        Q1 = train_raw[col].quantile(0.25)
        Q3 = train_raw[col].quantile(0.75)
        iqr_clip_bounds[col] = Q3 + 1.5 * (Q3 - Q1)

print('[train 기준 IQR 클리핑 상한값]')
for col, upper in iqr_clip_bounds.items():
    print(f'  {col}: {upper:.1f}')

In [ ]:
def _classify_treatment(x):
    """특정 시술 유형 → 5그룹 (규칙 기반, 누수 없음)"""
    t = str(x).upper().strip()
    if 'BLASTOCYST' in t: return 'Blastocyst_Transfer'
    elif 'ICSI'      in t: return 'ICSI'
    elif 'IVF'       in t: return 'IVF'
    elif 'IUI'       in t: return 'IUI'
    else:                  return 'Unknown'

# 시술 분류 그룹 생성 (fit 전 필요)
_tmp_treatment = train_raw['특정 시술 유형'].fillna('').apply(_classify_treatment)

# OneHotEncoder: train에서만 fit
OHE_COLS = ['시술_분류_그룹']  # 전처리 함수 내부에서 생성한 뒤 인코딩
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')
ohe.fit(_tmp_treatment.to_frame(name='시술_분류_그룹'))

print('[OHE 피팅 완료]')
print('생성될 컬럼:', ohe.get_feature_names_out(['시술_분류_그룹']).tolist())

In [ ]:
def preprocess_features(df, is_train: bool = True):
    """
    Parameters
    ----------
    df       : 원본 데이터프레임 (train 또는 test)
    is_train : True이면 타깃 컬럼을 보존, False이면 무시

    Returns
    -------
    pd.DataFrame — 전처리 완료, 원본 df 변경 없음
    """
    df = df.copy()

    # ──────────────────────────────────────────────────────────
    # STEP 0. 불필요 컬럼 제거
    # ──────────────────────────────────────────────────────────
    df = df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns], errors='ignore')

    # ──────────────────────────────────────────────────────────
    # STEP 1. 결측 이진 플래그 (STEP 11 fillna 전에 반드시 실행)
    # 결측 = '해당 시술 미수행'이라는 강력한 예측 신호를 보존
    # ──────────────────────────────────────────────────────────
    for col in ISNA_FLAG_COLS:
        if col in df.columns:
            df[f'{col}_isna'] = df[col].isna().astype(np.int8)

    # ──────────────────────────────────────────────────────────
    # STEP 2. 유전 검사 시행 여부 통합 (3개 컬럼 → 1개 이진)
    # ──────────────────────────────────────────────────────────
    present_genetic = [c for c in GENETIC_TEST_COLS if c in df.columns]
    df['유전검사_시행여부'] = (
        df[present_genetic].notna().any(axis=1) if present_genetic else False
    ).astype(np.int8)

    # ──────────────────────────────────────────────────────────
    # STEP 3. 나이 피처 (하드코딩 룩업)
    # Age_Median     : 구간 → 수치 중앙값
    # Age_Risk_Group : Normal / High_Early / High_Extreme / High_Reversal / Unknown
    # 원본 나이 컬럼 제거 (파생 피처와 colsample에서 희석 방지)
    # ──────────────────────────────────────────────────────────
    if '시술 당시 나이' in df.columns:
        df['Age_Median'] = (
            df['시술 당시 나이']
            .map({k: v['median'] for k, v in AGE_INFO.items()})
            .fillna(AGE_MEDIAN_FILLNA)  # '알 수 없음' → train 중앙값 (하드코딩)
        )
        df['Age_Risk_Group'] = (
            df['시술 당시 나이']
            .map({k: v['risk'] for k, v in AGE_INFO.items()})
            .fillna('Unknown')
        )
        df = df.drop(columns=['시술 당시 나이'])

    # ──────────────────────────────────────────────────────────
    # STEP 4. 시술 유형 5그룹 분류 (규칙 기반, 누수 없음)
    # ──────────────────────────────────────────────────────────
    if '특정 시술 유형' in df.columns:
        df['시술_분류_그룹'] = df['특정 시술 유형'].fillna('').apply(_classify_treatment)
        df = df.drop(columns=['특정 시술 유형'])

    # ──────────────────────────────────────────────────────────
    # STEP 5. 배아 잉여율
    # (총 생성 배아 수 - 이식된 배아 수) / 총 생성 배아 수
    # STEP 11 fillna 전에 실행 (원본 NaN이 살아있어야 safe_total이 정확함)
    # ──────────────────────────────────────────────────────────
    if '총 생성 배아 수' in df.columns and '이식된 배아 수' in df.columns:
        total    = df['총 생성 배아 수']
        transfer = df['이식된 배아 수'].fillna(0)
        safe_tot = total.where(total > 0, np.nan)
        df['배아_잉여율'] = ((safe_tot - transfer) / safe_tot).clip(0, 1)

    # ──────────────────────────────────────────────────────────
    # STEP 6. 수정률
    # 혼합된 난자 수 / 수집된 신선 난자 수
    # ──────────────────────────────────────────────────────────
    if '수집된 신선 난자 수' in df.columns and '혼합된 난자 수' in df.columns:
        collected      = df['수집된 신선 난자 수']
        mixed          = df['혼합된 난자 수'].fillna(0)
        safe_collected = collected.where(collected > 0, np.nan)
        df['수정률'] = (mixed / safe_collected).clip(0, 1)

    # ──────────────────────────────────────────────────────────
    # STEP 7. 로그 변환 (왜도 큰 배아/난자 수치)
    # STEP 11 fillna 전에 실행 (NaN → log1p(0)=0으로 자연 처리)
    # ──────────────────────────────────────────────────────────
    for col in LOG_COLS:
        if col in df.columns:
            df[f'{col}_log'] = np.log1p(df[col].fillna(0))

    # ──────────────────────────────────────────────────────────
    # STEP 8. IQR 클리핑 (train 기준값 재사용 → 누수 없음)
    # ──────────────────────────────────────────────────────────
    for col, upper in iqr_clip_bounds.items():
        if col in df.columns:
            df[f'{col}_clip'] = df[col].fillna(0).clip(upper=upper)

    # ──────────────────────────────────────────────────────────
    # STEP 9. 저장 배아 보유 여부 / 기증 정자 / 주기 구분
    # ──────────────────────────────────────────────────────────
    if '저장된 배아 수' in df.columns:
        df['저장배아_보유여부'] = (df['저장된 배아 수'].fillna(0) > 0).astype(np.int8)

    if '기증자 정자와 혼합된 난자 수' in df.columns:
        df['기증정자_사용여부'] = (df['기증자 정자와 혼합된 난자 수'].fillna(0) > 0).astype(np.int8)

    if '배아 해동 경과일' in df.columns:
        df['주기_구분'] = df['배아 해동 경과일'].isna().map(
            {True: '신선주기', False: '동결주기'}
        )

    # ──────────────────────────────────────────────────────────
    # STEP 10. 총 시술 횟수 (문자열 '1회' → 정수)
    # ──────────────────────────────────────────────────────────
    def _parse_count(col_name):
        if col_name not in df.columns:
            return pd.Series(0, index=df.index, dtype=np.int16)
        return (
            df[col_name].astype(str)
            .str.extract(r'(\d+)', expand=False)
            .fillna('0').astype(np.int16)
        )
    df['총_시술횟수'] = _parse_count('IVF 시술 횟수') + _parse_count('DI 시술 횟수')

    # ──────────────────────────────────────────────────────────
    # STEP 11. 난자 출처 기반 피처
    # ──────────────────────────────────────────────────────────
    if '난자 출처' in df.columns:
        df['기증난자_여부'] = (df['난자 출처'] == '기증 제공').astype(np.int8)

    # ──────────────────────────────────────────────────────────
    # STEP 12. 교호작용 피처 (행 단위 연산, 누수 없음)
    # ──────────────────────────────────────────────────────────
    age_med = df.get('Age_Median', pd.Series(AGE_MEDIAN_FILLNA, index=df.index))
    total_emb = df.get('총 생성 배아 수', pd.Series(0, index=df.index)).fillna(0)

    # 기증 난자 × 나이 (고령 기증 난자의 역전 효과 명시적 학습)
    if '기증난자_여부' in df.columns:
        df['기증난자×나이'] = df['기증난자_여부'] * age_med
        df['자가난자×나이'] = (1 - df['기증난자_여부']) * age_med

    # 나이 × 배아 수
    df['Age×배아수'] = age_med * total_emb

    # 초고위험군 교호작용
    is_extreme = df.get('Age_Risk_Group', pd.Series('', index=df.index)).isin(
        ['High_Extreme', 'High_Reversal']
    )
    if '기증난자_여부' in df.columns:
        df['초고위험_기증난자_조합'] = (is_extreme & (df['기증난자_여부'] == 1)).astype(np.int8)
    # 동결 배아 사용 여부: 배아 해동 경과일 isna 플래그 활용
    frozen_flag_col = '배아 해동 경과일_isna'
    if frozen_flag_col in df.columns:
        df['고령_동결배아_조합'] = (
            is_extreme & (df[frozen_flag_col] == 0)  # isna=0 → 경과일 있음 → 동결 배아 사용
        ).astype(np.int8)

    # 정상군 + 첫 시술 교호작용
    is_normal = df.get('Age_Risk_Group', pd.Series('', index=df.index)) == 'Normal'
    df['정상군_첫시술'] = (is_normal & (df['총_시술횟수'] == 0)).astype(np.int8)

    # ──────────────────────────────────────────────────────────
    # STEP 13. 멀티라벨 분해 (하드코딩 KNOWN_REASONS → 차원 불변)
    # ──────────────────────────────────────────────────────────
    MULTILABEL_COL = '배아 생성 주요 이유'
    if MULTILABEL_COL in df.columns:
        split_series = (
            df[MULTILABEL_COL].fillna('')
            .apply(lambda x: [v.strip() for v in x.split(',') if v.strip()])
        )
        for reason in KNOWN_REASONS:
            df[f'목적_{reason}'] = split_series.apply(lambda lst: int(reason in lst)).astype(np.int8)
        df = df.drop(columns=[MULTILABEL_COL])

    # ──────────────────────────────────────────────────────────
    # STEP 14. OneHotEncoder 적용 (train fit 완료된 ohe 재사용)
    # unseen category → 자동 0 처리 (handle_unknown='ignore')
    # ──────────────────────────────────────────────────────────
    if '시술_분류_그룹' in df.columns:
        ohe_arr = ohe.transform(df[['시술_분류_그룹']])
        ohe_df  = pd.DataFrame(
            ohe_arr,
            columns=ohe.get_feature_names_out(['시술_분류_그룹']),
            index=df.index
        ).astype(np.int8)
        df = pd.concat([df.drop(columns=['시술_분류_그룹']), ohe_df], axis=1)

    # ──────────────────────────────────────────────────────────
    # STEP 15. 수치형 결측 → -999
    # 트리가 -999를 별도 분기로 학습 → 구조적 결측 패턴 포착
    # STEP 1~14 파생변수 계산 완료 후 실행
    # ──────────────────────────────────────────────────────────
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if is_train and TARGET in num_cols:
        num_cols.remove(TARGET)
    df[num_cols] = df[num_cols].fillna(NUMERIC_FILL)

    # ──────────────────────────────────────────────────────────
    # STEP 16. 범주형 결측 → '알 수 없음' + category dtype
    # LightGBM은 category dtype으로 최적 분할, CatBoost는 str로 재변환 후 사용
    # ──────────────────────────────────────────────────────────
    obj_cols = [c for c in df.select_dtypes(include='object').columns
                if c != TARGET]
    for col in obj_cols:
        df[col] = df[col].fillna('알 수 없음').astype('category')

    return df

print('preprocess_features 정의 완료')

In [ ]:
def verify_no_leakage(train_raw, test_raw):
    """전처리 결과 9개 항목 자동 검증"""
    train_p = preprocess_features(train_raw, is_train=True)
    test_p  = preprocess_features(test_raw,  is_train=False)

    train_feat = [c for c in train_p.columns if c != TARGET]
    test_feat  = [c for c in test_p.columns  if c != TARGET]
    only_train = set(train_feat) - set(test_feat)
    only_test  = set(test_feat)  - set(train_feat)

    print('=' * 60)
    print('[1] 컬럼 구조 일치')
    if not only_train and not only_test:
        print('  PASS: train/test 피처 컬럼 완전 일치')
    else:
        if only_train: print(f'  WARN train only: {only_train}')
        if only_test:  print(f'  WARN test only:  {only_test}')

    print('\n[2] 결측 플래그(_isna) 생성')
    for col in ISNA_FLAG_COLS:
        flag = f'{col}_isna'
        if col in train_raw.columns:
            ok = (flag in train_p.columns) and (flag in test_p.columns)
            print(f'  {"PASS" if ok else "FAIL"}: {flag}')

    print('\n[3] 수치형 NaN 잔존 확인')
    num_tr = train_p.select_dtypes(include=[np.number])
    num_te = test_p.select_dtypes(include=[np.number])
    # 타깃 제외
    if TARGET in num_tr.columns: num_tr = num_tr.drop(columns=[TARGET])
    null_tr = num_tr.isna().sum().sum()
    null_te = num_te.isna().sum().sum()
    print(f'  {"PASS" if (null_tr == 0 and null_te == 0) else "FAIL"}: '
          f'train NaN={null_tr}, test NaN={null_te}')

    print('\n[4] object dtype 잔존 확인')
    obj_tr = [c for c in train_p.columns if train_p[c].dtype == 'object' and c != TARGET]
    obj_te = [c for c in test_p.columns  if test_p[c].dtype  == 'object']
    print(f'  {"PASS" if (not obj_tr and not obj_te) else "FAIL"}: '
          f'train={obj_tr}, test={obj_te}')

    print('\n[5] 주요 파생 변수 존재')
    check_cols = [
        'Age_Median', 'Age_Risk_Group', '유전검사_시행여부',
        '배아_잉여율', '수정률', '저장배아_보유여부', '총_시술횟수',
        '기증난자_여부', '기증난자×나이', '자가난자×나이',
        '초고위험_기증난자_조합', '정상군_첫시술',
    ]
    for col in check_cols:
        ok = (col in train_p.columns) and (col in test_p.columns)
        print(f'  {"PASS" if ok else "FAIL"}: {col}')

    print('\n[6] 원본 나이 컬럼 제거')
    removed = ('시술 당시 나이' not in train_p.columns and
               '시술 당시 나이' not in test_p.columns)
    print(f'  {"PASS" if removed else "FAIL"}: 시술 당시 나이 제거됨')

    print('\n[7] IQR 상한값 고정 여부 (train 기준 변수 존재)')
    print(f'  {"PASS" if iqr_clip_bounds else "FAIL"}: iqr_clip_bounds 항목 수={len(iqr_clip_bounds)}')

    print('\n[8] OHE train fit 여부')
    print(f'  PASS: ohe categories={[c.tolist() for c in ohe.categories_]}')

    print('\n[9] 재처리 동등성 (동일 입력 → 동일 출력)')
    try:
        pd.testing.assert_frame_equal(train_p, preprocess_features(train_raw, is_train=True))
        print('  PASS: 재처리 결과 동일')
    except AssertionError as e:
        print(f'  FAIL: {e}')

    print('\n' + '=' * 60)
    print(f'피처 수: train={len(train_feat)}, test={len(test_feat)}')
    new_feats = [c for c in train_feat if c not in train_raw.columns]
    print(f'신규 파생 피처 ({len(new_feats)}개):')
    for c in new_feats:
        print(f'  {c}')
    print('=' * 60)

verify_no_leakage(train_raw, test_raw)

In [ ]:
train_p = preprocess_features(train_raw, is_train=True)
test_p  = preprocess_features(test_raw,  is_train=False)

feat_cols = [c for c in train_p.columns if c != TARGET]
X      = train_p[feat_cols]
y      = train_p[TARGET].astype(int)
X_test = test_p[feat_cols]

print(f'X      : {X.shape}')
print(f'X_test : {X_test.shape}')
print(f'y 분포  : {dict(y.value_counts())}')
print(f'scale_pos_weight: {(y==0).sum()/(y==1).sum():.2f}')

In [ ]:
def run_lgbm_oof(X, y, X_test, params, n_splits=5, tag='lgbm'):
    scale_pos_weight = (y == 0).sum() / (y == 1).sum()
    full_params = {
        **params,
        'objective': 'binary', 'metric': 'auc',
        'scale_pos_weight': scale_pos_weight,
        'random_state': RANDOM_STATE, 'n_jobs': -1, 'verbose': -1,
    }
    cat_feats = [c for c in X.columns if X[c].dtype.name == 'category']
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof_preds  = np.zeros(len(y))
    test_preds = np.zeros(len(X_test))
    models = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        model = lgb.LGBMClassifier(**full_params)
        model.fit(
            X.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
            callbacks=[
                lgb.early_stopping(100, verbose=False),
                lgb.log_evaluation(500),
            ],
            categorical_feature=cat_feats,
        )
        oof_preds[val_idx]  = model.predict_proba(X.iloc[val_idx])[:, 1]
        test_preds         += model.predict_proba(X_test)[:, 1] / n_splits
        auc = roc_auc_score(y.iloc[val_idx], oof_preds[val_idx])
        print(f'  [{tag}] Fold {fold+1}: AUC={auc:.4f}  iter={model.best_iteration_}')
        models.append(model)

    oof_auc = roc_auc_score(y, oof_preds)
    print(f'\n  [{tag}] OOF AUC={oof_auc:.4f}  (vs 0.7404: {oof_auc-0.7404:+.4f})')
    return models, oof_preds, test_preds, oof_auc

print('run_lgbm_oof 정의 완료')

In [ ]:
def run_catboost_oof(X, y, X_test, params=None, n_splits=5, tag='catboost'):
    scale_pos_weight = (y == 0).sum() / (y == 1).sum()
    cat_feats_idx = [i for i, c in enumerate(X.columns) if X[c].dtype.name == 'category']

    # CatBoost는 category dtype 대신 str 선호
    X_cb      = X.copy()
    X_test_cb = X_test.copy()
    for col in X.columns:
        if X[col].dtype.name == 'category':
            X_cb[col]      = X_cb[col].astype(str)
            X_test_cb[col] = X_test_cb[col].astype(str)

    default_params = {
        'iterations': 2000, 'learning_rate': 0.05, 'depth': 6,
        'l2_leaf_reg': 3, 'scale_pos_weight': scale_pos_weight,
        'eval_metric': 'AUC', 'random_seed': RANDOM_STATE,
        'verbose': False, 'early_stopping_rounds': 100,
    }
    if params:
        default_params.update(params)

    skf        = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    oof_preds  = np.zeros(len(y))
    test_preds = np.zeros(len(X_test))
    models     = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_cb, y)):
        model = CatBoostClassifier(**default_params)
        model.fit(
            X_cb.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=(X_cb.iloc[val_idx], y.iloc[val_idx]),
            cat_features=cat_feats_idx,
        )
        oof_preds[val_idx]  = model.predict_proba(X_cb.iloc[val_idx])[:, 1]
        test_preds         += model.predict_proba(X_test_cb)[:, 1] / n_splits
        auc = roc_auc_score(y.iloc[val_idx], oof_preds[val_idx])
        print(f'  [{tag}] Fold {fold+1}: AUC={auc:.4f}')
        models.append(model)

    oof_auc = roc_auc_score(y, oof_preds)
    print(f'\n  [{tag}] OOF AUC={oof_auc:.4f}  (vs 0.7404: {oof_auc-0.7404:+.4f})')
    return models, oof_preds, test_preds, oof_auc

print('run_catboost_oof 정의 완료')

In [ ]:
def run_lgbm_optuna(X, y, n_trials=50, cv_splits=3):
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    scale_pos_weight = (y == 0).sum() / (y == 1).sum()
    cat_feats = [c for c in X.columns if X[c].dtype.name == 'category']
    skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)

    def objective(trial):
        params = {
            'objective': 'binary', 'metric': 'auc', 'n_estimators': 3000,
            'learning_rate':     trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
            'num_leaves':        trial.suggest_int('num_leaves', 16, 128),
            'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
            'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
            'subsample_freq':    1,
            'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.4, 1.0),
            'reg_alpha':         trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda':        trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'scale_pos_weight':  scale_pos_weight,
            'random_state': RANDOM_STATE, 'n_jobs': -1, 'verbose': -1,
        }
        aucs = []
        for tr_idx, val_idx in skf.split(X, y):
            model = lgb.LGBMClassifier(**params)
            model.fit(
                X.iloc[tr_idx], y.iloc[tr_idx],
                eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
                callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
                categorical_feature=cat_feats,
            )
            aucs.append(roc_auc_score(y.iloc[val_idx],
                                      model.predict_proba(X.iloc[val_idx])[:, 1]))
        return np.mean(aucs)

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )
    print(f'[LGBM Optuna] {n_trials}회 탐색 시작...')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print(f'\n[완료] Best {cv_splits}-fold AUC: {study.best_value:.4f}')
    for k, v in study.best_params.items():
        print(f'  {k:25s}: {v}')
    # Optuna가 탐색할 때 objective 함수 안에서 subsample_freq=1을 사용했음
    # 그런데 반환값에 subsample_freq가 빠지면, Optuna가 평가한 조합과
    #   실제 최종 학습 조합이 달라지는 문제가 생김
    # LightGBM에서 subsample(행 샘플링)은 subsample_freq가 0이면
    #   적용되지 않으므로, 반드시 함께 반환해야 함
    #  * 모델 평가에 쓴 파라미터와 실제 학습 파라미터가 일치해야 검증 점수를 신뢰할 수 있음
    return {**study.best_params, 'n_estimators': 3000, 'subsample_freq': 1}


def run_catboost_optuna(X, y, n_trials=30, cv_splits=3):
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    scale_pos_weight = (y == 0).sum() / (y == 1).sum()
    cat_feats_idx = [i for i, c in enumerate(X.columns) if X[c].dtype.name == 'category']
    X_cb = X.copy()
    for col in X.columns:
        if X[col].dtype.name == 'category':
            X_cb[col] = X_cb[col].astype(str)

    skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)

    def objective(trial):
        params = {
            'iterations':   trial.suggest_int('iterations', 500, 3000, step=500),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'depth':        trial.suggest_int('depth', 4, 8),
            'l2_leaf_reg':  trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
            'scale_pos_weight': scale_pos_weight,
            'eval_metric': 'AUC', 'random_seed': RANDOM_STATE,
            'verbose': False, 'early_stopping_rounds': 100,
        }
        aucs = []
        for tr_idx, val_idx in skf.split(X_cb, y):
            model = CatBoostClassifier(**params)
            model.fit(
                X_cb.iloc[tr_idx], y.iloc[tr_idx],
                eval_set=(X_cb.iloc[val_idx], y.iloc[val_idx]),
                cat_features=cat_feats_idx,
            )
            aucs.append(roc_auc_score(y.iloc[val_idx],
                                      model.predict_proba(X_cb.iloc[val_idx])[:, 1]))
        return np.mean(aucs)

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )
    print(f'[CatBoost Optuna] {n_trials}회 탐색 시작...')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    print(f'\n[완료] Best {cv_splits}-fold AUC: {study.best_value:.4f}')
    for k, v in study.best_params.items():
        print(f'  {k:25s}: {v}')
    return study.best_params

print('Optuna 함수 정의 완료')

In [ ]:
# v4 Optuna에서 찾은 파라미터로 빠른 베이스라인 확인
lgbm_v4_params = {
    'n_estimators':      3000,
    'learning_rate':     0.032275921555211334,
    'num_leaves':        32,
    'min_child_samples': 31,
    'subsample':         0.9443563832121702,
    'subsample_freq':    1,
    'colsample_bytree':  0.4660046628277167,
    'reg_alpha':         3.546463779196944,
    'reg_lambda':        2.1863992476849217,
}

lgbm_models, lgbm_oof, lgbm_test, lgbm_auc = run_lgbm_oof(
    X, y, X_test, lgbm_v4_params, tag='lgbm_v4_params'
)

In [ ]:


lgbm_best_params = run_lgbm_optuna(X, y, n_trials=50, cv_splits=3)

lgbm_opt_models, lgbm_opt_oof, lgbm_opt_test, lgbm_opt_auc = run_lgbm_oof(
    X, y, X_test, lgbm_best_params, tag='lgbm_optuna'
)

In [ ]:
# 시간 절약을 위해 3-fold 30회로 탐색 후 5-fold 최종 학습
cat_best_params = run_catboost_optuna(X, y, n_trials=30, cv_splits=3)

cat_models, cat_oof, cat_test, cat_auc = run_catboost_oof(
    X, y, X_test, params=cat_best_params, tag='catboost_optuna'
)

In [ ]:
# 1. CatBoost 기본 학습
cat_models, cat_oof, cat_test, cat_auc = run_catboost_oof(
    X, y, X_test, params=None, tag='catboost'
)

In [ ]:
def find_best_weights(oof_list, test_list, y, labels=None, step=0.05):
    """
    두 모델의 가중치를 step 단위로 그리드 탐색.
    w1 + w2 = 1.0 조건에서 최적 비율을 찾습니다.
    """
    assert len(oof_list) == 2, '현재 2-모델 앙상블만 지원'
    if labels is None:
        labels = [f'model_{i}' for i in range(len(oof_list))]

    best_auc, best_w = 0, (0.5, 0.5)
    results = []
    for w1 in np.arange(0.0, 1.0 + step, step):
        w1 = round(w1, 4)
        w2 = round(1.0 - w1, 4)
        oof_blend = w1 * oof_list[0] + w2 * oof_list[1]
        auc = roc_auc_score(y, oof_blend)
        results.append((w1, w2, auc))
        if auc > best_auc:
            best_auc, best_w = auc, (w1, w2)

    # 결과 시각화
    weights = [r[0] for r in results]
    aucs    = [r[2] for r in results]
    plt.figure(figsize=(10, 4))
    plt.plot(weights, aucs, marker='o', markersize=4)
    plt.axvline(best_w[0], color='red', linestyle='--', label=f'최적 w1={best_w[0]:.2f}')
    plt.xlabel(f'{labels[0]} 가중치 (w1)')
    plt.ylabel('OOF AUC')
    plt.title('앙상블 가중치 탐색')
    plt.legend()
    plt.tight_layout()
    plt.show()

    print(f'\n[앙상블 최적 가중치]')
    print(f'  {labels[0]}: {best_w[0]:.2f}  |  {labels[1]}: {best_w[1]:.2f}')
    print(f'  OOF AUC: {best_auc:.4f}  (vs 0.7404: {best_auc-0.7404:+.4f})')

    # 최적 가중치로 test 예측
    test_blend = best_w[0] * test_list[0] + best_w[1] * test_list[1]
    return test_blend, best_w, best_auc


# 앙상블 가중치 탐색
test_ensemble, best_weights, ensemble_auc = find_best_weights(
    oof_list  = [lgbm_opt_oof, cat_oof],
    test_list = [lgbm_opt_test, cat_test],
    y         = y,
    labels    = ['LGBM_Optuna', 'CatBoost'],
)

In [ ]:
# 저장
submission = pd.read_csv(DATA_DIR + 'sample_submission.csv')
submission['probability'] = test_ensemble
submission.to_csv('submission_v5_final.csv', index=False)
print(f"저장 완료 | 앙상블 AUC: {ensemble_auc:.4f}")

In [ ]:
print('=' * 60)
print('           v5 최종 결과 요약')
print('=' * 60)
print(f'  LGBM (v4 params) OOF AUC : {lgbm_auc:.4f}')
print(f'  LGBM Optuna      OOF AUC : {lgbm_opt_auc:.4f}')
print(f'  CatBoost Optuna  OOF AUC : {cat_auc:.4f}')
print(f'  앙상블 (최적 가중치) AUC  : {ensemble_auc:.4f}')
print(f'  가중치: LGBM={best_weights[0]:.2f}, CatBoost={best_weights[1]:.2f}')
print(f'  v4 대비 개선: {ensemble_auc - 0.7404:+.4f}')
print('=' * 60)

In [ ]:
submission = pd.read_csv(DATA_DIR + 'sample_submission.csv')
submission['probability'] = test_ensemble
submission.to_csv('submission_v5_ensemble.csv', index=False)
print(f'저장 완료: submission_v5_ensemble.csv')
print(submission.head(3))

In [ ]:
# 단일 모델 제출 파일도 함께 저장 (비교용)
submission_lgbm = pd.read_csv(DATA_DIR + 'sample_submission.csv')
submission_lgbm['probability'] = lgbm_opt_test
submission_lgbm.to_csv('submission_v5_lgbm.csv', index=False)



In [ ]:
submission_cat = pd.read_csv(DATA_DIR + 'sample_submission.csv')
submission_cat['probability'] = cat_test
submission_cat.to_csv('submission_v5_catboost.csv', index=False)

print('저장 완료: submission_v5_lgbm.csv, submission_v5_catboost.csv')

In [ ]:
# 마지막 fold 모델의 피처 중요도 (gain 기준)
last_lgbm = lgbm_opt_models[-1]
importance_df = pd.DataFrame({
    'feature':    X.columns,
    'importance': last_lgbm.booster_.feature_importance(importance_type='gain'),
}).sort_values('importance', ascending=False).head(30)

plt.figure(figsize=(10, 8))
plt.barh(importance_df['feature'][::-1], importance_df['importance'][::-1])
plt.title('LightGBM Top-30 Feature Importance (gain)')
plt.tight_layout()
plt.show()

print('\nTop-10 피처:')
print(importance_df[['feature', 'importance']].head(10).to_string(index=False))